# Notebook 09: Gold Layer — Batch Inference + Maintenance Schedule
## Purpose: Load Production models, run batch inference, write Gold tables
## Input:  workspace.predictive_maintenance.silver_machine_enriched
## Output:
##   - gold_failure_predictions  — failure probability per machine
##   - gold_maintenance_schedule — ranked maintenance priority list

## Why Gold Layer?
Silver = cleaned + enriched features. Still technical.
Gold = business-ready outputs. A maintenance engineer reads Gold.
Gold answers: "Which machines do I fix today and in what order?"


In [0]:
%pip install xgboost mlflow scikit-learn
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
import mlflow
import mlflow.xgboost
import pandas as pd
import numpy as np
from mlflow.tracking import MlflowClient
import warnings
warnings.filterwarnings('ignore')

# Load enriched silver table
df = spark.table("workspace.predictive_maintenance.silver_machine_enriched")

# Feature columns
EXCLUDE_COLS = [
    "unit_id", "cycle", "RUL", "fail_30", "fail_15",
    "source_dataset", "setting_1", "setting_2", "setting_3"
]
FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Loaded: {df.count():,} rows | {len(df.columns)} columns")
print(f"Feature columns: {len(FEATURE_COLS)}")

In [0]:
classifier = mlflow.xgboost.load_model(
    "models:/workspace.default.predmaint-classifier@champion"
)
print("Classifier loaded: predmaint-classifier@champion")

In [0]:
rul_model = mlflow.xgboost.load_model(
    "models:/workspace.default.predmaint-rul@champion"
)
print("RUL model loaded: predmaint-rul@champion")

In [0]:
# Load all data to pandas
pdf_all = df.select(
    ["unit_id", "cycle", "RUL", "fail_30"] + FEATURE_COLS
).toPandas().fillna(0)

# Get max cycle per machine
max_cycles = pdf_all.groupby('unit_id')['cycle'].max()
pdf_all['max_cycle']      = pdf_all['unit_id'].map(max_cycles)
pdf_all['lifecycle_pct']  = pdf_all['cycle'] / pdf_all['max_cycle']

# Pick row closest to 80% lifecycle per machine
# This simulates a real fleet where machines are at different health stages
pdf_all['dist_to_80'] = abs(pdf_all['lifecycle_pct'] - 0.80)
pdf = pdf_all.loc[
    pdf_all.groupby('unit_id')['dist_to_80'].idxmin()
].copy()

X = pdf[FEATURE_COLS]

# Run inference
pdf['failure_probability'] = classifier.predict_proba(X)[:, 1]
pdf['failure_predicted']   = classifier.predict(X)
pdf['rul_predicted']       = np.maximum(0, rul_model.predict(X))

print(f"Snapshot: {len(pdf):,} machines at 80% lifecycle")
print(f"\nFailure probability spread:")
print(f"Min:  {pdf['failure_probability'].min():.4f}")
print(f"Max:  {pdf['failure_probability'].max():.4f}")
print(f"Mean: {pdf['failure_probability'].mean():.4f}")
print(f"\nRUL spread:")
print(f"Min:  {pdf['rul_predicted'].min():.1f}")
print(f"Max:  {pdf['rul_predicted'].max():.1f}")
print(f"Mean: {pdf['rul_predicted'].mean():.1f}")

In [0]:
# Use percentile-based thresholds — works with any dataset
rul_25th = pdf['rul_predicted'].quantile(0.25)
rul_50th = pdf['rul_predicted'].quantile(0.50)

print(f"RUL thresholds:")
print(f"25th percentile (RED threshold):    {rul_25th:.1f} cycles")
print(f"50th percentile (YELLOW threshold): {rul_50th:.1f} cycles")

def assign_risk_zone(rul):
    if rul <= rul_25th:
        return 'RED'
    elif rul <= rul_50th:
        return 'YELLOW'
    else:
        return 'GREEN'

pdf['risk_zone'] = pdf['rul_predicted'].apply(assign_risk_zone)

print("\n=== FLEET RISK SUMMARY ===")
zone_counts = pdf['risk_zone'].value_counts()
for zone in ['RED', 'YELLOW', 'GREEN']:
    count = zone_counts.get(zone, 0)
    print(f"{zone:>8}: {count} machines")
print(f"\nTotal machines: {len(pdf)}")

In [0]:
predictions_sdf = spark.createDataFrame(
    pdf[['unit_id', 'cycle', 'RUL',
         'failure_probability', 'failure_predicted',
         'rul_predicted', 'risk_zone', 'fail_30']]
)

predictions_sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.predictive_maintenance.gold_failure_predictions"
    )

spark.sql("""
    OPTIMIZE workspace.predictive_maintenance.gold_failure_predictions
    ZORDER BY (unit_id, cycle)
""")

count = spark.table(
    "workspace.predictive_maintenance.gold_failure_predictions"
).count()
print(f"gold_failure_predictions written: {count:,} rows")

In [0]:
predictions_sdf.createOrReplaceTempView("new_predictions")

spark.sql("""
    MERGE INTO workspace.predictive_maintenance.gold_failure_predictions 
    AS target
    USING new_predictions AS source
    ON target.unit_id = source.unit_id 
    AND target.cycle  = source.cycle
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

final_count = spark.table(
    "workspace.predictive_maintenance.gold_failure_predictions"
).count()

print(f"MERGE INTO complete: {final_count:,} rows")
print("Production pattern: UPSERT not overwrite")

In [0]:
# Already one row per machine from Cell 6
at_risk = pdf.copy()
at_risk = at_risk.sort_values('rul_predicted')
at_risk['urgency_rank'] = range(1, len(at_risk) + 1)

at_risk['recommended_action'] = at_risk['risk_zone'].map({
    'RED':    'IMMEDIATE — Schedule within 24 hours',
    'YELLOW': 'PLANNED — Schedule within 1 week',
    'GREEN':  'MONITOR — No action needed'
})

at_risk['estimated_cost_savings'] = at_risk['risk_zone'].map({
    'RED':    45000,
    'YELLOW': 45000,
    'GREEN':  0
})

print(f"Total machines in schedule: {len(at_risk)}")
print(f"\nTop 10 most urgent:")
print(at_risk[['unit_id', 'rul_predicted', 'risk_zone',
               'urgency_rank', 'recommended_action']] \
      .head(10).to_string(index=False))

In [0]:
maint_sdf = spark.createDataFrame(
    at_risk[[
        'unit_id', 'rul_predicted', 'failure_probability',
        'risk_zone', 'urgency_rank',
        'recommended_action', 'estimated_cost_savings'
    ]]
)

maint_sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.predictive_maintenance.gold_maintenance_schedule"
    )

count = spark.table(
    "workspace.predictive_maintenance.gold_maintenance_schedule"
).count()

print(f"gold_maintenance_schedule written: {count} machines")

# Total cost savings
total_savings = at_risk['estimated_cost_savings'].sum()
red_count     = len(at_risk[at_risk['risk_zone'] == 'RED'])
yellow_count  = len(at_risk[at_risk['risk_zone'] == 'YELLOW'])

print(f"\n=== BUSINESS IMPACT ===")
print(f"RED    machines: {red_count}")
print(f"YELLOW machines: {yellow_count}")
print(f"Total estimated savings: ${total_savings:,.0f}")
print(f"\nThis is your video closing statement:")
print(f"'This system identified {red_count + yellow_count} at-risk")
print(f" machines — preventing an estimated")
print(f" ${total_savings:,.0f} in unplanned downtime costs.'")

In [0]:
maint_sdf = spark.createDataFrame(
    at_risk[[
        'unit_id', 'rul_predicted', 'failure_probability',
        'risk_zone', 'urgency_rank',
        'recommended_action', 'estimated_cost_savings'
    ]]
)

maint_sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.predictive_maintenance.gold_maintenance_schedule"
    )

count = spark.table(
    "workspace.predictive_maintenance.gold_maintenance_schedule"
).count()

total_savings = at_risk['estimated_cost_savings'].sum()
red_count     = len(at_risk[at_risk['risk_zone'] == 'RED'])
yellow_count  = len(at_risk[at_risk['risk_zone'] == 'YELLOW'])
green_count   = len(at_risk[at_risk['risk_zone'] == 'GREEN'])

print(f"gold_maintenance_schedule written: {count} machines")
print(f"\n=== FLEET HEALTH SUMMARY ===")
print(f"RED    (critical):  {red_count} machines")
print(f"YELLOW (warning):   {yellow_count} machines")
print(f"GREEN  (healthy):   {green_count} machines")
print(f"\nTotal savings: ${total_savings:,.0f}")
print(f"\nVideo closing line:")
print(f"'System identified {red_count + yellow_count} at-risk machines")
print(f" — preventing ${total_savings:,.0f} in unplanned downtime.'")

In [0]:
print("=== ALL TABLES IN PROJECT ===")
spark.sql(
    "SHOW TABLES IN workspace.predictive_maintenance"
).show(20, truncate=False)